# Multimodal Sentiment Classification with CLIP (Colab Subset Version)

This notebook adapts the multimodal meme sentiment classification pipeline for **Google Colab** using a **reduced number of images and a reduced number of sentiment labels**.

### Key Modifications from the Original Notebook
1. **Google Colab Compatibility**: Mounts Google Drive (or extracts a local `.zip` archive uploaded to Colab) instead of using `/kaggle/input` paths.
2. **Reduced Number of Labels (Binary Classification)**: Merges sentiment annotations into **2 classes** (`positive` vs. `non_positive`, combining `negative` and `neutral`) rather than 3 classes.
3. **Reduced Dataset Size (Subsampling)**: Limits the total number of samples via stratified sampling (`MAX_SAMPLES = 1200`) so experiments run quickly on Colab GPUs.
4. **All-English Documentation**: All markdown sections, explanations, and inline code comments are written in English.

## 1. Environment Setup and Data Loading in Google Colab

Choose one of the two options below to load your reduced dataset in Google Colab:
- **Option A (`USE_GDRIVE = True`)**: Mounts Google Drive and reads `labels.csv` and image files from your Drive folder.
- **Option B (`USE_GDRIVE = False`)**: Extracts a `.zip` archive uploaded to `/content` and reads from the extracted directory.

Update `CSV_PATH`, `IMAGES_DIR`, or `ZIP_PATH` as needed for your dataset layout.

In [21]:
# Install required packages in Colab if not already installed
!pip install -q torch torchvision transformers scikit-learn

import os
import copy
import zipfile
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import train_test_split
from transformers import CLIPProcessor, CLIPModel

# Allow truncated images to load without raising errors
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Choose how to load the reduced dataset in Google Colab
USE_GDRIVE = True

if USE_GDRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CSV_PATH = "/content/drive/MyDrive/test2/labels.csv"
    IMAGES_DIR = "/content/drive/MyDrive/test2/images"
else:
    ZIP_PATH = "/content/memotion_subset.zip"
    EXTRACT_DIR = "/content/memotion_subset"
    if os.path.exists(ZIP_PATH) and not os.path.exists(EXTRACT_DIR):
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(EXTRACT_DIR)
    CSV_PATH = os.path.join(EXTRACT_DIR, "labels.csv")
    IMAGES_DIR = os.path.join(EXTRACT_DIR, "images")

print("CSV path:", CSV_PATH)
print("Images directory:", IMAGES_DIR)

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSV path: /content/drive/MyDrive/test2/labels.csv
Images directory: /content/drive/MyDrive/test2/images


## 2. Load and Clean Dataset

In this step, we:
1. Load the dataset CSV containing `image_name`, `text_corrected`, and `overall_sentiment`.
2. Drop rows with missing values or blank text.
3. Verify each image file with `PIL.Image.open(...).verify()` and skip corrupted or missing images.

In [22]:
# Subsampling and reproducibility settings
MAX_SAMPLES = 1200  # Target number of samples to retain (set to None to keep all valid samples)
RANDOM_SEED = 42    # Seed used for stratified subsampling and data splits

# Load metadata and filter out missing or empty text/labels
df = pd.read_csv(CSV_PATH)
df = df[["image_name", "text_corrected", "overall_sentiment"]].dropna(
    subset=["text_corrected", "overall_sentiment"]
)
df["text_corrected"] = df["text_corrected"].astype(str)
df = df[df["text_corrected"].str.strip() != ""]

# Verify image integrity and discard unreadable/corrupt images
bad_images = 0
valid_indices = []
for idx, row in df.iterrows():
    img_path = os.path.join(IMAGES_DIR, str(row["image_name"]))
    try:
        with Image.open(img_path) as im:
            im.verify()
        valid_indices.append(idx)
    except Exception:
        bad_images += 1

df = df.loc[valid_indices].reset_index(drop=True)
print(f"Skipped {bad_images} corrupt/missing images. Valid images available: {len(df)}")

Skipped 0 corrupt/missing images. Valid images available: 7


## 3. Reduce Number of Labels and Subsample Image Count

### Reducing the Number of Labels (2 Classes)
We reduce the sentiment label space to **2 classes**:
- `"positive"` (`label = 0`): `positive` and `very_positive`
- `"non_positive"` (`label = 1`): all remaining labels (`neutral`, `negative`, `very_negative`)

### Reducing the Number of Images
When `MAX_SAMPLES` is smaller than the number of valid rows, we perform stratified sampling (`train_test_split`) to keep a smaller subset while maintaining class balance.

In [23]:
def reduce_sentiment_label(label: str) -> str:
    """Map original sentiment labels into 2 reduced classes: positive vs. non_positive."""
    normalized = str(label).strip().lower()
    if normalized in ["positive", "very_positive"]:
        return "positive"
    return "non_positive"

# Convert to reduced 2-class labels
df["reduced_sentiment"] = df["overall_sentiment"].apply(reduce_sentiment_label)
label2id = {"positive": 0, "non_positive": 1}
id2label = {idx: name for name, idx in label2id.items()}
df["label"] = df["reduced_sentiment"].map(label2id)

# Reduce actual image/sample count via stratified sampling
if MAX_SAMPLES is not None and len(df) > MAX_SAMPLES:
    df, _ = train_test_split(
        df,
        train_size=MAX_SAMPLES,
        stratify=df["label"],
        random_state=RANDOM_SEED,
    )
    df = df.reset_index(drop=True)

print(f"Final dataset size: {len(df)} samples | Number of labels: {len(label2id)}")
print("\nClass distribution in reduced dataset:")
print(df["reduced_sentiment"].value_counts(normalize=True))

Final dataset size: 7 samples | Number of labels: 2

Class distribution in reduced dataset:
reduced_sentiment
positive        0.571429
non_positive    0.428571
Name: proportion, dtype: float64


## 4. Stratified Train / Validation / Test Split

We split the reduced dataset into **80% train**, **10% validation**, and **10% test** subsets using stratification (`stratify=df["label"]`) so each split reflects the same label proportions.

In [24]:
# Split 80% train and 20% temporary (validation + test)
try:
    train_df, temp_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=RANDOM_SEED,
    )
except ValueError:
    # Fallback to non-stratified split if dataset is too small
    print("Warning: Dataset too small for stratified split. Falling back to standard split for train/temp.")
    train_df, temp_df = train_test_split(
        df,
        test_size=0.2,
        random_state=RANDOM_SEED,
    )

# Split the 20% temporary set evenly into 10% validation and 10% test
try:
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df["label"],
        random_state=RANDOM_SEED,
    )
except ValueError:
    # Fallback to non-stratified split if temp_df is too small
    print("Warning: Temporary set too small for stratified split. Falling back to standard split for val/test.")
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=RANDOM_SEED,
    )

print(f"Split sizes -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Split sizes -> Train: 5 | Val: 1 | Test: 1


## 5. CLIP Processor, Custom Dataset, and DataLoaders

We initialize `CLIPProcessor` from `openai/clip-vit-base-patch32` to preprocess RGB images (`pixel_values`) and tokenize text (`input_ids`, `attention_mask`) with a maximum sequence length of `77` tokens.

In [25]:
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)


class CLIPReducedMemeDataset(Dataset):
    """Custom PyTorch Dataset pairing preprocessed CLIP image and text inputs with class labels."""

    def __init__(self, dataframe: pd.DataFrame, images_dir: str, processor: CLIPProcessor, max_length: int = 77):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = images_dir
        self.processor = processor
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, str(row["image_name"]))
        image = Image.open(img_path).convert("RGB")
        text = str(row["text_corrected"])
        label = torch.tensor(int(row["label"]), dtype=torch.long)

        # Encode image and text jointly into PyTorch tensors
        encoded = self.processor(
            text=[text],
            images=[image],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": label,
        }


BATCH_SIZE = 16

train_dataset = CLIPReducedMemeDataset(train_df, IMAGES_DIR, processor)
val_dataset = CLIPReducedMemeDataset(val_df, IMAGES_DIR, processor)
test_dataset = CLIPReducedMemeDataset(test_df, IMAGES_DIR, processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 6. Multimodal CLIP Classifier Architecture

`CLIPClassifier` wraps `CLIPModel`, extracts both image embeddings (`image_embeds`) and text embeddings (`text_embeds`), concatenates them along the feature dimension, and passes the combined vector through a small MLP head to produce logits for the reduced number of classes (`num_labels = 2`).

In [26]:
class CLIPClassifier(nn.Module):
    """Multimodal classifier fusing CLIP image and text embeddings for sentiment prediction."""

    def __init__(self, model_name: str, num_labels: int = 2, freeze_clip: bool = False):
        super().__init__()
        self.clip_model = CLIPModel.from_pretrained(model_name)
        self.num_labels = num_labels

        # Optionally freeze CLIP weights to reduce compute and memory usage
        if freeze_clip:
            for param in self.clip_model.parameters():
                param.requires_grad = False

        fused_dim = self.clip_model.config.projection_dim * 2
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels),
        )

    def forward(self, pixel_values: torch.Tensor, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.clip_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
        )
        fused = torch.cat([outputs.image_embeds, outputs.text_embeds], dim=1)
        logits = self.classifier(fused)
        return logits


NUM_LABELS = len(label2id)
model = CLIPClassifier(
    model_name=CLIP_MODEL_NAME,
    num_labels=NUM_LABELS,
    freeze_clip=False,
).to(DEVICE)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 7. Training Loop, Early Stopping, and Evaluation

We train with `AdamW` and `CrossEntropyLoss`, reduce the learning rate with `ReduceLROnPlateau` when validation loss plateaus, and stop early if validation loss fails to improve for `PATIENCE` epochs. Finally, we restore the best validation checkpoint and evaluate on the test set.

In [27]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1
)
criterion = nn.CrossEntropyLoss()


def run_epoch(model: nn.Module, dataloader: DataLoader, is_train: bool = False):
    """Execute a single training or evaluation epoch and return loss and metrics."""
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    grad_context = torch.enable_grad() if is_train else torch.no_grad()
    with grad_context:
        for batch in dataloader:
            pixel_values = batch["pixel_values"].to(DEVICE)
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            logits = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

            preds = logits.argmax(dim=1).detach().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.detach().cpu().numpy())

            if is_train:
                loss.backward()
                optimizer.step()

    avg_loss = total_loss / len(dataloader.dataset)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    return avg_loss, acc, prec, rec, f1, all_labels, all_preds


EPOCHS = 5
PATIENCE = 2
best_val_loss = float("inf")
no_improve = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_prec, train_rec, train_f1, _, _ = run_epoch(
        model, train_loader, is_train=True
    )
    val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = run_epoch(
        model, val_loader, is_train=False
    )

    scheduler.step(val_loss)

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(
        f"  Train -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | "
        f"P: {train_prec:.4f} | R: {train_rec:.4f} | F1: {train_f1:.4f}"
    )
    print(
        f"  Val   -> Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
        f"P: {val_prec:.4f} | R: {val_rec:.4f} | F1: {val_f1:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

# Load best validation weights before final evaluation
if best_state is not None:
    model.load_state_dict(best_state)

test_loss, test_acc, test_prec, test_rec, test_f1, y_true, y_pred = run_epoch(
    model, test_loader, is_train=False
)

print("\n--- Test Results (Colab Reduced Dataset & Labels) ---")
print(f"Test Loss:       {test_loss:.4f}")
print(f"Test Accuracy:   {test_acc:.4f}")
print(f"Macro Precision: {test_prec:.4f}")
print(f"Macro Recall:    {test_rec:.4f}")
print(f"Macro F1:        {test_f1:.4f}")
print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_LABELS)),
        target_names=[id2label[i] for i in range(NUM_LABELS)],
        zero_division=0,
    )
)


Epoch 1/5
  Train -> Loss: 0.6845 | Acc: 0.6000 | P: 0.3000 | R: 0.5000 | F1: 0.3750
  Val   -> Loss: 0.7216 | Acc: 0.0000 | P: 0.0000 | R: 0.0000 | F1: 0.0000

Epoch 2/5
  Train -> Loss: 0.6650 | Acc: 0.6000 | P: 0.3000 | R: 0.5000 | F1: 0.3750
  Val   -> Loss: 0.7266 | Acc: 0.0000 | P: 0.0000 | R: 0.0000 | F1: 0.0000

Epoch 3/5
  Train -> Loss: 0.6471 | Acc: 0.8000 | P: 0.8750 | R: 0.7500 | F1: 0.7619
  Val   -> Loss: 0.7325 | Acc: 0.0000 | P: 0.0000 | R: 0.0000 | F1: 0.0000
Early stopping triggered.

--- Test Results (Colab Reduced Dataset & Labels) ---
Test Loss:       0.6650
Test Accuracy:   1.0000
Macro Precision: 1.0000
Macro Recall:    1.0000
Macro F1:        1.0000

Classification Report:
              precision    recall  f1-score   support

    positive       1.00      1.00      1.00         1
non_positive       0.00      0.00      0.00         0

    accuracy                           1.00         1
   macro avg       0.50      0.50      0.50         1
weighted avg       1